Blend: does combining the GNN and XGBoost beat either one alone? (Phase 5.4, score-level blend, no retraining)

Inputs (attach as Kaggle inputs):
  - papertrail-processed          (transactions.parquet: to join transaction ids to accounts, times and amounts)
  - papertrail-scores             (a private dataset you create from reports/gnn_scores.parquet and reports/xgb_scores.parquet)
  - ibm-transactions-for-anti-money-laundering-aml  (HI-Small_Patterns.txt, for recall per typology)
CPU, Internet ON, DAGSHUB_TOKEN secret attached. Runs in about a minute. Save & Run All, or just run the cells.

Method (all choices on VALIDATION only; test is scored once):
  1. Convert each model's probability to a logit and standardise it with that model's validation mean and std
     (so the two scores are on the same scale, and test uses the validation constants).
  2. Blend = w * gnn + (1 - w) * xgb, with w picked from 0, 0.1, ..., 1 by validation PR-AUC.
  3. The alert threshold is the validation F1-optimal one, as for every model.
Reported per seed pair (gnn_s0 with xgb_s0, ...) as mean +- std, and for the 3-seed ensembles (gnn_mean, xgb_mean), which is the
recipe the demo will use. Also: recall per typology, and how many laundering transactions each model catches that the other misses.



In [1]:
!pip install -q git+https://github.com/abhibarnwal707/PaperTrail.git
import os
from kaggle_secrets import UserSecretsClient
os.environ["DAGSHUB_USER_TOKEN"] = UserSecretsClient().get_secret("DAGSHUB_TOKEN")
import dagshub, mlflow
dagshub.init(repo_owner="abhibarnwal707", repo_name="papertrail", mlflow=True)


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.1/51.1 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.0/274.0 kB 7.4 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 93.4 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 93.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 62.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.5/231.5 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228

Accessing as abhibarnwal707

Initialized MLflow to track repo "abhibarnwal707/papertrail"

Repository abhibarnwal707/papertrail initialized!

In [2]:
import time as _t; print("[cell 2: load] started", _t.strftime("%H:%M:%S"))
import glob
import json
import os

import numpy as np
import polars as pl

from papertrail.metrics import best_f1_threshold, evaluate, pr_auc


def find(name):
    hits = glob.glob(f"/kaggle/input/datasets/abhibarnwal707/**/{name}", recursive=True)
    if not hits:
        raise FileNotFoundError(f"{name} not found under /kaggle/input: attach the dataset that holds it")
    return hits[0]


gnn = pl.read_parquet(find("gnn_scores.parquet"))
xgb = pl.read_parquet(find("xgb_scores.parquet"))
SEEDS = sorted(int(c[5:]) for c in gnn.columns if c.startswith("gnn_s"))
m = gnn.join(xgb.select(["txn_id"] + [c for c in xgb.columns if c.startswith("xgb_")]), on="txn_id", how="inner")
assert m.height == gnn.height == xgb.height, "the two score files do not cover the same transactions"
assert (gnn["label"].to_numpy() == m["label"].to_numpy()).all()
val, test = m.filter(pl.col("split") == "val"), m.filter(pl.col("split") == "test")
y_val, y_test = val["label"].to_numpy(), test["label"].to_numpy()
print(f"seeds {SEEDS} | val {val.height} rows / {int(y_val.sum())} positives | test {test.height} rows / {int(y_test.sum())} positives")


[cell 2: load] started 12:58:35
seeds [0, 1, 2] | val 965524 rows / 1036 positives | test 862792 rows / 956 positives


In [3]:
import time as _t; print("[cell 3: helpers] started", _t.strftime("%H:%M:%S"))
KEEP = ["pr_auc", "f1", "recall_at_1pct_fpr", "precision_at_1000"]
GRID = np.linspace(0, 1, 11)


def logit(p):
    p = np.clip(p, 1e-7, 1 - 1e-7)
    return np.log(p / (1 - p))


def standardise(p_val, p_test):
    """Logit, then standardise with the VALIDATION mean and std (applied unchanged to test)."""
    lv, lt = logit(p_val), logit(p_test)
    mu, sd = float(lv.mean()), float(lv.std())
    return (lv - mu) / sd, (lt - mu) / sd, mu, sd


def run_pair(g_val, g_test, x_val, x_test):
    zg_v, zg_t, g_mu, g_sd = standardise(g_val, g_test)
    zx_v, zx_t, x_mu, x_sd = standardise(x_val, x_test)
    w = float(max(GRID, key=lambda a: pr_auc(y_val, a * zg_v + (1 - a) * zx_v)))
    arms = {"gnn": (zg_v, zg_t), "xgb": (zx_v, zx_t), "blend": (w * zg_v + (1 - w) * zx_v, w * zg_t + (1 - w) * zx_t)}
    out = {}
    for name, (sv, st) in arms.items():
        thr = float(best_f1_threshold(y_val, sv))
        e = evaluate(y_test, st, thr)
        out[name] = {**{k: float(e[k]) for k in KEEP}, "val_pr_auc": float(pr_auc(y_val, sv)), "threshold": thr}
    consts = {"w_gnn": w, "gnn_mu": g_mu, "gnn_sd": g_sd, "xgb_mu": x_mu, "xgb_sd": x_sd, "threshold": out["blend"]["threshold"]}
    return out, consts, arms


[cell 3: helpers] started 13:00:00


In [4]:
import time as _t; print("[cell 4: blend] started", _t.strftime("%H:%M:%S"))
per_seed, seed_w = {}, {}
for s in SEEDS:
    per_seed[s], c, _ = run_pair(val[f"gnn_s{s}"].to_numpy(), test[f"gnn_s{s}"].to_numpy(), val[f"xgb_s{s}"].to_numpy(), test[f"xgb_s{s}"].to_numpy())
    seed_w[s] = c["w_gnn"]

ens, consts, arms = run_pair(val["gnn_mean"].to_numpy(), test["gnn_mean"].to_numpy(), val["xgb_mean"].to_numpy(), test["xgb_mean"].to_numpy())
print(f"{'':10s}" + "".join(f"{k:>26s}" for k in KEEP))
summary = {}
for name in ("xgb", "gnn", "blend"):
    summary[name] = {k: (float(np.mean([per_seed[s][name][k] for s in SEEDS])), float(np.std([per_seed[s][name][k] for s in SEEDS]))) for k in KEEP}
    print(f"{name:10s}" + "".join(f"{summary[name][k][0]:>17.3f} +-{summary[name][k][1]:.3f}" for k in KEEP))
print(f"\n3-seed ensembles (average probability per model, then blend), the recipe the demo will use:")
for name in ("xgb", "gnn", "blend"):
    print(f"  {name:6s}" + "".join(f"  {k} {ens[name][k]:.3f}" for k in KEEP))
print(f"blend weight on the GNN (validation-chosen): per seed {seed_w}, ensemble {consts['w_gnn']}")


[cell 4: blend] started 13:00:33
                              pr_auc                        f1        recall_at_1pct_fpr         precision_at_1000
xgb                   0.436 +-0.004            0.506 +-0.007            0.659 +-0.007            0.443 +-0.007
gnn                   0.255 +-0.025            0.297 +-0.023            0.675 +-0.034            0.290 +-0.020
blend                 0.497 +-0.016            0.542 +-0.013            0.724 +-0.021            0.475 +-0.007

3-seed ensembles (average probability per model, then blend), the recipe the demo will use:
  xgb     pr_auc 0.442  f1 0.512  recall_at_1pct_fpr 0.657  precision_at_1000 0.446
  gnn     pr_auc 0.281  f1 0.315  recall_at_1pct_fpr 0.708  precision_at_1000 0.305
  blend   pr_auc 0.505  f1 0.550  recall_at_1pct_fpr 0.729  precision_at_1000 0.483
blend weight on the GNN (validation-chosen): per seed {0: 0.4, 1: 0.4, 2: 0.30000000000000004}, ensemble 0.4


In [9]:
KNOWN = {   # exact locations first: a full recursive search of /kaggle/input can take over a minute
    "transactions.parquet": "/kaggle/input/datasets/abhibarnwal707/papertrail-processed/",
    "gnn_scores.parquet": "/kaggle/input/datasets/abhibarnwal707/papertrail-scores/",
    "xgb_scores.parquet": "/kaggle/input/datasets/abhibarnwal707/papertrail-scores/",
    "HI-Small_Patterns.txt": "/kaggle/input/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml/",
}
def find(name):
    exact = KNOWN.get(name, "") + name
    if os.path.exists(exact):
        return exact
    for pattern in (f"/kaggle/input/datasets/*/*/{name}", f"/kaggle/input/*/{name}", f"/kaggle/input/**/{name}"):
        hits = glob.glob(pattern, recursive=True)       # any owner/dataset name, cheapest search first
        if hits:
            return hits[0]
    raise FileNotFoundError(f"{name} not found under /kaggle/input: attach the dataset that holds it")


In [10]:
import time as _t; print("[cell 5: typology] started", _t.strftime("%H:%M:%S"))
sv_g, st_g = arms["gnn"]
sv_x, st_x = arms["xgb"]
# At a fixed false-alarm budget (1% of normal test transactions flagged), which positives does each model catch?
neg = y_test == 0
cut_g = np.quantile(st_g[neg], 0.99)
cut_x = np.quantile(st_x[neg], 0.99)
caught_g, caught_x = (st_g >= cut_g) & (y_test == 1), (st_x >= cut_x) & (y_test == 1)
n_pos = int(y_test.sum())
overlap = {"both": int((caught_g & caught_x).sum()), "gnn_only": int((caught_g & ~caught_x).sum()),
           "xgb_only": int((caught_x & ~caught_g).sum()), "neither": int((~caught_g & ~caught_x & (y_test == 1)).sum())}
print(f"At a 1% false-alarm budget on test ({n_pos} laundering transactions):", overlap)

tx = pl.read_parquet(find("transactions.parquet")).select(["txn_id", "ts", "src", "dst", "amt_paid"])
rows, typ = [], None
with open(find("HI-Small_Patterns.txt")) as fh:
    for line in fh:
        line = line.strip()
        if line.startswith("BEGIN"):
            typ = line.split(" - ")[1].split(":")[0].strip()
        elif line and not line.startswith("END"):
            f = line.split(",")
            rows.append((typ, f[0], f"{int(f[1])}_{f[2]}", f"{int(f[3])}_{f[4]}", float(f[7])))
key = ["ts", "src", "dst", "amt_paid"]
pat = (pl.DataFrame(rows, schema=["typology"] + key, orient="row").with_columns(pl.col("ts").str.strptime(pl.Datetime, "%Y/%m/%d %H:%M"))
         .unique(subset=key, keep="first", maintain_order=True))
typ_of = test.select("txn_id").join(tx, on="txn_id").join(pat, on=key, how="left").select("txn_id", pl.col("typology").fill_null("UNMAPPED"))
typ_arr = typ_of["typology"].to_numpy()
thr = {n: ens[n]["threshold"] for n in ("xgb", "gnn", "blend")}
scores_test = {"xgb": st_x, "gnn": st_g, "blend": arms["blend"][1]}
typ_rows = []
for t in sorted(set(typ_arr[y_test == 1])):
    sel = (y_test == 1) & (typ_arr == t)
    typ_rows.append({"typology": t, "n": int(sel.sum()), **{f"recall_{n}": float((scores_test[n][sel] >= thr[n]).mean()) for n in ("xgb", "gnn", "blend")}})
typ_tbl = pl.DataFrame(typ_rows).sort("n", descending=True)
pl.Config.set_tbl_rows(20)
print(typ_tbl.with_columns(pl.col(pl.Float64).round(3)))


[cell 5: typology] started 13:11:42
At a 1% false-alarm budget on test (956 laundering transactions): {'both': 550, 'gnn_only': 127, 'xgb_only': 78, 'neither': 201}
shape: (9, 5)
┌────────────────┬─────┬────────────┬────────────┬──────────────┐
│ typology       ┆ n   ┆ recall_xgb ┆ recall_gnn ┆ recall_blend │
│ ---            ┆ --- ┆ ---        ┆ ---        ┆ ---          │
│ str            ┆ i64 ┆ f64        ┆ f64        ┆ f64          │
╞════════════════╪═════╪════════════╪════════════╪══════════════╡
│ UNMAPPED       ┆ 371 ┆ 0.003      ┆ 0.038      ┆ 0.003        │
│ GATHER-SCATTER ┆ 127 ┆ 0.74       ┆ 0.543      ┆ 0.78         │
│ SCATTER-GATHER ┆ 110 ┆ 0.618      ┆ 0.527      ┆ 0.718        │
│ STACK          ┆ 84  ┆ 0.512      ┆ 0.31       ┆ 0.524        │
│ FAN-OUT        ┆ 65  ┆ 0.6        ┆ 0.6        ┆ 0.708        │
│ FAN-IN         ┆ 57  ┆ 0.702      ┆ 0.544      ┆ 0.772        │
│ CYCLE          ┆ 55  ┆ 0.527      ┆ 0.327      ┆ 0.582        │
│ BIPARTITE      ┆ 44  ┆ 0.63

In [11]:
import time as _t; print("[cell 6: save] started", _t.strftime("%H:%M:%S"))
out = {"seeds": SEEDS, "per_seed": {str(s): per_seed[s] for s in SEEDS}, "mean_std": summary, "ensemble": ens, "blend_recipe": consts, "seed_weights": {str(k): v for k, v in seed_w.items()},
       "overlap_at_1pct_fpr": overlap, "typology_recall": typ_tbl.to_dicts()}
with open("/kaggle/working/blend_summary.json", "w") as fh:
    json.dump(out, fh, indent=2)
typ_tbl.write_csv("/kaggle/working/typology_recall_models.csv")
pl.DataFrame({"txn_id": test["txn_id"].to_numpy(), "label": y_test, "xgb_z": st_x, "gnn_z": st_g, "blend_z": arms["blend"][1]}
             ).write_parquet("/kaggle/working/blend_scores_test.parquet")
with mlflow.start_run(run_name="blend"):
    mlflow.log_params({"w_gnn": consts["w_gnn"], "seeds": len(SEEDS), "method": "logit standardised on validation; weight by validation PR-AUC"})
    mlflow.log_metrics({f"{n}_{k}_{stat}": v for n in summary for k, (a, b) in summary[n].items() for v, stat in ((a, "mean"), (b, "std"))})
    mlflow.log_metrics({f"ensemble_{n}_{k}": ens[n][k] for n in ens for k in KEEP})
    for p in ("/kaggle/working/blend_summary.json", "/kaggle/working/typology_recall_models.csv"):
        mlflow.log_artifact(p)
print("Download blend_summary.json and typology_recall_models.csv into PaperTrail/reports/.")


[cell 6: save] started 13:11:47
🏃 View run blend at: https://dagshub.com/abhibarnwal707/papertrail.mlflow/#/experiments/0/runs/d413dea253c84c3da2b190bdf4d4d93d
🧪 View experiment at: https://dagshub.com/abhibarnwal707/papertrail.mlflow/#/experiments/0
Download blend_summary.json and typology_recall_models.csv into PaperTrail/reports/.
